# Notebook 5: Evaluation & Prompt Engineering Experiments

## Purpose
This notebook systematically compares prompt versions and measures the
impact of each prompt engineering technique on agent behavior.

## Metrics tracked
- **Hallucination rate** — did the model suggest ingredients not in the pantry?
- **Expiry prioritization** — did the top suggestion use the soonest-expiring item?
- **Tool call efficiency** — how many tool calls per turn?
- **Output structure** — did the response follow the required format?
- **Grounding compliance** — did the model verify the pantry before suggesting?

> **Prerequisite:** Run notebooks 01–04 first.

---

In [ ]:
import mysql.connector
import json
import os
import time
import sql_openai_config
from datetime import date, datetime
from pathlib import Path
from openai import OpenAI

# Paths only used for saving JSON results
PROJECT_ROOT = Path(os.getcwd()).parent
JSON_PATH    = PROJECT_ROOT / "data" / "pantry_items.json"

MYSQL_CONFIG = sql_openai_config.get_mysql_config()


# API key: keep using your env var or this line if you prefer
os.environ["OPENAI_API_KEY"] = sql_openai_config.get_openai()

client = OpenAI()

def get_connection():
    return mysql.connector.connect(**MYSQL_CONFIG)

print("Setup complete (MySQL).")


Setup complete (MySQL).


In [7]:
# ── Redefine tools inline ─────────────────────────────────────────────────────

def get_pantry_items() -> list:
    conn = get_connection()
    cur  = conn.cursor()
    cur.execute(
        "SELECT name, category, quantity, unit, expiry_date "
        "FROM pantry ORDER BY expiry_date ASC"
    )
    rows = cur.fetchall()
    conn.close()
    today = date.today()
    return [
        {
            "name": n,
            "category": c,
            "quantity": q,
            "unit": u,
            "expiry_date": e,
            "days_until_expiry": (
                datetime.strptime(str(e), "%Y-%m-%d").date() - today
            ).days if e else None,
        }
        for n, c, q, u, e in rows
    ]


def get_at_risk_items(threshold_days: int = 3) -> list:
    return [
        {**i, "status": "EXPIRED" if i["days_until_expiry"] < 0 else "AT RISK"}
        for i in get_pantry_items()
        if i["days_until_expiry"] is not None
        and i["days_until_expiry"] <= threshold_days
    ]


def add_pantry_item(name, category, quantity, unit, expiry_date):
    conn = get_connection()
    cur  = conn.cursor()
    cur.execute(
        "INSERT INTO pantry (name, category, quantity, unit, expiry_date) "
        "VALUES (%s,%s,%s,%s,%s)",
        (name, category, quantity, unit, expiry_date),
    )
    conn.commit(); conn.close()
    return f"Added '{name}'."


def remove_pantry_item(name):
    conn = get_connection()
    cur  = conn.cursor()
    cur.execute(
        "DELETE FROM pantry WHERE LOWER(name)=LOWER(%s)",
        (name,),
    )
    conn.commit()
    affected = cur.rowcount
    conn.close()
    return f"Removed '{name}'." if affected > 0 else f"'{name}' not found."


def update_quantity(name, new_quantity):
    if new_quantity <= 0:
        return remove_pantry_item(name)
    conn = get_connection()
    cur  = conn.cursor()
    cur.execute(
        "UPDATE pantry SET quantity=%s WHERE LOWER(name)=LOWER(%s)",
        (new_quantity, name),
    )
    conn.commit()
    affected = cur.rowcount
    conn.close()
    return f"Updated '{name}' to {new_quantity}." if affected > 0 else f"'{name}' not found."


TOOL_DEFINITIONS = [
    {'type':'function','function':{'name':'get_pantry_items','description':'Get all pantry items with expiry info. Call before any recipe suggestion.','parameters':{'type':'object','properties':{},'required':[]}}},
    {'type':'function','function':{'name':'get_at_risk_items','description':'Get items expiring within threshold_days (default 3).','parameters':{'type':'object','properties':{'threshold_days':{'type':'integer'}},'required':[]}}},
    {'type':'function','function':{'name':'add_pantry_item','description':'Add new ingredient.','parameters':{'type':'object','properties':{'name':{'type':'string'},'category':{'type':'string'},'quantity':{'type':'number'},'unit':{'type':'string'},'expiry_date':{'type':'string'}},'required':['name','category','quantity','unit','expiry_date']}}},
    {'type':'function','function':{'name':'remove_pantry_item','description':'Remove ingredient.','parameters':{'type':'object','properties':{'name':{'type':'string'}},'required':['name']}}},
    {'type':'function','function':{'name':'update_quantity','description':'Update remaining quantity.','parameters':{'type':'object','properties':{'name':{'type':'string'},'new_quantity':{'type':'number'}},'required':['name','new_quantity']}}}
]

TOOL_MAP = {
    "get_pantry_items"   : lambda a: get_pantry_items(),
    "get_at_risk_items"  : lambda a: get_at_risk_items(**a),
    "add_pantry_item"    : lambda a: add_pantry_item(**a),
    "remove_pantry_item" : lambda a: remove_pantry_item(**a),
    "update_quantity"    : lambda a: update_quantity(**a),
}

print("Tools ready.")


Tools ready.


## 5.1 Evaluation runner

This function runs a test prompt against a given system prompt,
executes tool calls, and collects metrics automatically.

In [8]:
def run_eval(system_prompt: str, user_message: str, label: str = '') -> dict:
    """
    Run one evaluation turn and return metrics.
    Returns a dict with: response, tool_calls_made, tools_used, duration_sec
    """
    messages = []
    if system_prompt:
        messages.append({'role': 'system', 'content': system_prompt})
    messages.append({'role': 'user', 'content': user_message})

    tools_used      = []
    tool_call_count = 0
    start           = time.time()

    while True:
        response = client.chat.completions.create(
            model='gpt-4o-mini', messages=messages,
            tools=TOOL_DEFINITIONS, tool_choice='auto'
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return {
                'label'           : label,
                'response'        : msg.content,
                'tool_calls_made' : tool_call_count,
                'tools_used'      : tools_used,
                'duration_sec'    : round(time.time() - start, 2)
            }

        messages.append(msg)
        for tc in msg.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)
            tool_call_count += 1
            tools_used.append(fn_name)
            result = TOOL_MAP[fn_name](fn_args) if fn_name in TOOL_MAP else 'ERROR'
            messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': json.dumps(result, default=str)})

print('Evaluation runner ready.')

Evaluation runner ready.


## 5.2 Define prompt versions for comparison

In [9]:
PROMPTS = {
    'V1_no_system': '',

    'V2_persona': """
You are the Smart Pantry Chef, a friendly kitchen assistant for zero-waste cooking.
Help users use up ingredients before they expire.
""",

    'V3_grounding': """
You are the Smart Pantry Chef, a friendly kitchen assistant for zero-waste cooking.
NEVER suggest an ingredient not confirmed in the pantry.
Always call get_pantry_items() or get_at_risk_items() before suggesting any recipe.
Prioritize ingredients closest to expiry.
""",

    'V4_cot': """
You are the Smart Pantry Chef, a friendly kitchen assistant for zero-waste cooking.
NEVER suggest an ingredient not confirmed in the pantry.

## Reasoning sequence — follow every time
Step 1: Call get_at_risk_items() to identify expiring ingredients.
Step 2: Call get_pantry_items() to see the full inventory.
Step 3: Match at-risk items to a viable recipe using confirmed ingredients only.
Step 4: Explain your reasoning before giving the recipe.
""",

    'V5_full': """
You are the Smart Pantry Chef, an AI kitchen assistant specialized in zero-waste cooking.

## Reasoning rules
1. NEVER suggest an ingredient not confirmed in the pantry. Always call tools first.
2. ALWAYS prioritize at-risk ingredients (expiring within 3 days).
3. Only list pantry-confirmed ingredients. Salt and water may be labeled (assumed).
4. After every recipe, offer to update the pantry inventory.

## Recipe output format
**Recipe:** [Name]
**Why this recipe?** [At-risk ingredients, days remaining]
**Ingredients from your pantry:** [list]
**Assumed staples:** (if any)
**Instructions:** 1-8 steps
**Pantry update:** Offer to update.

## Tone: warm, encouraging, practical.
"""
}

print(f'Defined {len(PROMPTS)} prompt versions: {list(PROMPTS.keys())}')

Defined 5 prompt versions: ['V1_no_system', 'V2_persona', 'V3_grounding', 'V4_cot', 'V5_full']


## 5.3 Run all experiments against standard test prompts

In [10]:
TEST_QUESTIONS = [
    'What should I cook tonight to avoid wasting anything?',
    'Suggest something using my dairy items.',
    'I just used 2 eggs. Please update my pantry.',
]

all_results = []

for q_idx, question in enumerate(TEST_QUESTIONS):
    print(f'\n{"─"*60}')
    print(f'TEST QUESTION {q_idx+1}: {question}')
    print(f'{"─"*60}')

    for version, prompt in PROMPTS.items():
        print(f'  Running {version}...')
        result = run_eval(prompt, question, label=version)
        result['question'] = question
        all_results.append(result)
        print(f'  Tools called: {result["tools_used"]} | Duration: {result["duration_sec"]}s')
        time.sleep(1)  # Avoid rate limits

print(f'\nAll experiments complete. Total runs: {len(all_results)}')


────────────────────────────────────────────────────────────
TEST QUESTION 1: What should I cook tonight to avoid wasting anything?
────────────────────────────────────────────────────────────
  Running V1_no_system...
  Tools called: ['get_at_risk_items'] | Duration: 9.34s
  Running V2_persona...
  Tools called: ['get_at_risk_items'] | Duration: 7.3s
  Running V3_grounding...
  Tools called: ['get_at_risk_items'] | Duration: 7.26s
  Running V4_cot...
  Tools called: ['get_at_risk_items', 'get_pantry_items'] | Duration: 10.19s
  Running V5_full...
  Tools called: ['get_pantry_items', 'get_at_risk_items'] | Duration: 16.1s

────────────────────────────────────────────────────────────
TEST QUESTION 2: Suggest something using my dairy items.
────────────────────────────────────────────────────────────
  Running V1_no_system...
  Tools called: ['get_pantry_items'] | Duration: 10.62s
  Running V2_persona...
  Tools called: ['get_pantry_items'] | Duration: 12.94s
  Running V3_grounding...
 

## 5.4 Metrics summary table

In [11]:
def check_grounding(tools_used: list) -> bool:
    """Did the agent verify the pantry before answering?"""
    return any(t in ('get_pantry_items', 'get_at_risk_items') for t in tools_used)

def check_expiry_priority(response: str, tools_used: list) -> bool:
    """Did the agent call get_at_risk_items (expiry-aware tool)?"""
    return 'get_at_risk_items' in tools_used

def check_structure(response: str) -> bool:
    """Does response contain the required format sections?"""
    if not response:
        return False
    required = ['Recipe', 'Ingredient', 'Instruction']
    return all(kw.lower() in response.lower() for kw in required)

# Group by question for cleaner output
for q_idx, question in enumerate(TEST_QUESTIONS):
    q_results = [r for r in all_results if r['question'] == question]
    print(f'\nQuestion {q_idx+1}: {question}')
    print(f"{'Version':<18} {'Tools called':>12} {'Grounded?':>10} {'Expiry?':>8} {'Structured?':>12} {'Time':>6}")
    print('-' * 72)
    for r in q_results:
        grounded   = '✓' if check_grounding(r['tools_used'])        else '✗'
        expiry     = '✓' if check_expiry_priority(r['response'], r['tools_used']) else '✗'
        structured = '✓' if check_structure(r['response'])          else '✗'
        print(f"{r['label']:<18} {r['tool_calls_made']:>12} {grounded:>10} {expiry:>8} {structured:>12} {r['duration_sec']:>5}s")


Question 1: What should I cook tonight to avoid wasting anything?
Version            Tools called  Grounded?  Expiry?  Structured?   Time
------------------------------------------------------------------------
V1_no_system                  1          ✓        ✓            ✗  9.34s
V2_persona                    1          ✓        ✓            ✗   7.3s
V3_grounding                  1          ✓        ✓            ✗  7.26s
V4_cot                        2          ✓        ✓            ✓ 10.19s
V5_full                       2          ✓        ✓            ✓  16.1s

Question 2: Suggest something using my dairy items.
Version            Tools called  Grounded?  Expiry?  Structured?   Time
------------------------------------------------------------------------
V1_no_system                  1          ✓        ✗            ✗ 10.62s
V2_persona                    1          ✓        ✗            ✓ 12.94s
V3_grounding                  1          ✓        ✗            ✓ 16.78s
V4_cot        

## 5.5 Print full responses for manual review

In [12]:
# Choose which question and version to inspect
TARGET_QUESTION = TEST_QUESTIONS[0]   # change index to inspect other questions
TARGET_VERSION  = 'V5_full'           # change to compare versions

match = [
    r for r in all_results
    if r['question'] == TARGET_QUESTION and r['label'] == TARGET_VERSION
]

if match:
    r = match[0]
    print(f'Question : {r["question"]}')
    print(f'Version  : {r["label"]}')
    print(f'Tools    : {r["tools_used"]}')
    print(f'Duration : {r["duration_sec"]}s')
    print(f'\nResponse:\n{"─"*60}')
    print(r['response'])
else:
    print('No match found. Check TARGET_QUESTION and TARGET_VERSION.')

Question : What should I cook tonight to avoid wasting anything?
Version  : V5_full
Tools    : ['get_pantry_items', 'get_at_risk_items']
Duration : 16.1s

Response:
────────────────────────────────────────────────────────────
Here's a great recipe to make use of some of your at-risk ingredients for tonight!

**Recipe:** Savory Salmon and Vegetable Skillet  
**Why this recipe?** At-risk ingredients: tomato (2 days), banana (2 days), whole wheat bread (2 days), croissants (2 days), milk (3 days), salmon fillet (3 days), lettuce romaine (3 days), mushrooms button (3 days), raspberries (3 days), sourdough bread (3 days).  

**Ingredients from your pantry:**  
- Salmon fillet (400 grams)  
- Lettuce romaine (2 heads)  
- Mushrooms button (400 grams)  
- Tomato (4 pieces)  
- Raspberries (250 grams)  
- Whole wheat bread (1 loaf) or croissants (6 pieces)  

**Assumed staples:** Salt, water, and any spices you prefer.

**Instructions:**  
1. Start by preheating a skillet over medium heat.  
2

## 5.6 Export results to JSON for write-up

In [13]:
output_path = PROJECT_ROOT / 'data' / 'experiment_results.json'

with open(output_path, 'w') as f:
    json.dump(all_results, f, indent=2, default=str)

print(f'Results saved to: {output_path}')
print(f'Total records   : {len(all_results)}')

Results saved to: /Users/nahlkhan/Desktop/smart_pantry/data/experiment_results.json
Total records   : 15
